# California Housing Price Prediction

An end-to-end regression pipeline on the 1990 California Census housing dataset.

**Pipeline overview:**
1. Load dataset directly from scikit-learn (no CSV needed)
2. Exploratory data analysis — null checks, feature inspection, spatial diversity
3. Feature engineering — merge external `ocean_proximity` column and encode categoricals
4. Visualization — boxplot by ocean proximity, income vs price jointplot
5. Linear regression — train/test split, fit, evaluate (MAE, RMSE), scatter plot

**Dataset:** 20,640 California block groups from the 1990 Census.  
**Target:** `MedHouseVal` — median house value in units of $100,000.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

---
## 1. Loading the Dataset

`fetch_california_housing()` downloads the dataset directly from scikit-learn's built-in collection — no manual CSV download needed. With `as_frame=True`, it returns a `Bunch` object where `.frame` is already a fully-formed Pandas DataFrame, so no extra conversion step is required.

Other scikit-learn toy datasets include `load_iris()` (flower classification) and `load_diabetes()` (disease progression regression).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Shape: {df.shape}")
df.head()

---
## 2. Exploratory Data Analysis

### 2.1 Feature overview

| Feature | Description |
|---|---|
| `MedInc` | Median household income (tens of thousands of dollars) |
| `HouseAge` | Median age of houses in the block |
| `AveRooms` | Average rooms per household |
| `AveBedrms` | Average bedrooms per household |
| `Population` | Total block population |
| `AveOccup` | Average household size |
| `Latitude` | Block centroid latitude |
| `Longitude` | Block centroid longitude |
| `MedHouseVal` | **Target** — median house value ($100,000s) |

In [ ]:
df.head(3)

### 2.2 Null check

A clean dataset requires no imputation before modelling.

In [ ]:
df.info()
print("\nNull counts:")
print(df.isnull().sum())

### 2.3 Geographic diversity

Checking how many distinct latitude/longitude coordinates exist — a proxy for spatial coverage across California.

In [ ]:
unique_lat = df['Latitude'].nunique()
unique_lon = df['Longitude'].nunique()

print(f"Unique latitudes : {unique_lat}")
print(f"Unique longitudes: {unique_lon}")
print(f"The dataset covers {unique_lat * unique_lon:,} possible coordinate combinations across California.")

---
## 3. Feature Engineering — Adding Ocean Proximity

The base dataset is missing `ocean_proximity` — a categorical feature indicating how close each block is to the ocean. It's provided as a separate CSV.

Two steps:
1. Merge it into the main DataFrame on row index (`axis=1`)
2. Encode the 5 string categories as integers using a mapping dictionary — regression requires numeric inputs

Note: a more robust approach for production would be `pd.get_dummies()` (one-hot encoding) to avoid implying ordinal relationships between categories. Label encoding is used here for simplicity.

In [ ]:
ocean_df = pd.read_csv("Ocean_Proximity.csv")

# Merge on row index (axis=1 = column-wise concat)
df['ocean_proximity'] = ocean_df['ocean_proximity']

print("Categories:", df['ocean_proximity'].unique())
print(f"Dataset now has {df.shape[1]} columns")

In [ ]:
# Label-encode: each unique category gets an integer code
keys = df['ocean_proximity'].unique()
values = range(len(keys))
proximity_map = {keys[i]: values[i] for i in range(len(keys))}

df['ocean_proximity'] = df['ocean_proximity'].map(proximity_map)

print("Encoding map:", proximity_map)
df[['ocean_proximity']].head()

---
## 4. Visualization

### 4.1 Ocean Proximity vs Median House Value

Do houses closer to the ocean cost more?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='ocean_proximity', y='MedHouseVal', data=df)
plt.title('Median House Value by Ocean Proximity', fontsize=14)
plt.xlabel('Ocean Proximity')
plt.ylabel('Median House Value ($100,000s)')
plt.xticks([0, 1, 2, 3, 4], ['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'])
plt.tight_layout()
plt.show()

print("""
Insight: A clear price gradient exists by location.
ISLAND properties have the highest median values, followed by coastal categories.
INLAND homes are significantly cheaper — roughly half the value of coastal blocks.
This confirms ocean proximity as a meaningful predictor for housing prices.
""")

### 4.2 Median Income vs Median House Value

Income is often the strongest predictor of house prices — visualising their joint distribution.

In [ ]:
sns.set(style="white", color_codes=True)
sns.jointplot(x='MedInc', y='MedHouseVal', data=df, kind='hex', color='orange', height=8)
plt.suptitle('Median Income vs Median House Value', y=1.02, fontsize=13)
plt.show()

print("""
Insight: Strong positive correlation between income and house value.
The density is highest in the low-income, low-value range (0-4 MedInc, 0-2 MedHouseVal).
A hard cap at MedHouseVal = 5.0 is visible — a data collection artifact from the 1990 census,
where values above $500,000 were top-coded to avoid outlier distortion.
""")

---
## 5. Linear Regression

### 5.1 Prepare features and target

In [ ]:
# Keep only numeric columns (drops any remaining object columns)
numeric_df = df.select_dtypes(include=['int64', 'float64'])

X = numeric_df.drop('MedHouseVal', axis=1)
y = numeric_df['MedHouseVal']

print(f"Features (X): {X.shape}  →  {list(X.columns)}")
print(f"Target   (y): {y.shape}")

### 5.2 Train / test split

80/20 split with `random_state=42` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Test samples     : {X_test.shape[0]:,}")

### 5.3 Train and evaluate

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE  : {mae:.4f}  (≈ ${mae * 100_000:,.0f} average error per prediction)")
print(f"RMSE : {rmse:.4f}  (penalises large errors more heavily than MAE)")
print()
print("""
Interpretation:
On average, the model's predictions are off by ~$53,000.
RMSE > MAE indicates some predictions have larger errors — 
likely driven by the top-coded values at $500,000 and geographic outliers.
A non-linear model (e.g. Random Forest) would likely reduce both metrics.
""")

### 5.4 Actual vs predicted scatter plot

Points on the dashed diagonal = perfect prediction. Spread around it = model error.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='teal', label='Predictions')
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red', linewidth=2, linestyle='--', label='Perfect prediction'
)
plt.title('Actual vs Predicted Median House Values', fontsize=14)
plt.xlabel('Actual Median House Value ($100,000s)')
plt.ylabel('Predicted Median House Value ($100,000s)')
plt.legend()
plt.tight_layout()
plt.show()

print("""
Insight: The model tracks the general trend well but over-predicts for the
top-coded $500,000 cluster (visible as a vertical band at x=5.0).
Linear regression cannot capture the non-linear relationships in this data —
especially the geographic clustering of high-value properties.
""")